# 01 — Load and prepare the data

**Stage:** obtaining and preparing the data.

This notebook:
1. reads the original UCI file `data/raw/bank-additional-full.csv`;
2. checks it (size, target, missing values);
3. rebuilds the **year** of each call, which the file does not contain;
4. assigns each call to a time period (train, validation, test 1, test 2);
5. saves the result to `data/processed/bank_with_dates.csv`, used by the later notebooks.

Source of the data: Moro, Cortez and Rita (2014), *A Data-Driven Approach to Predict the Success of Bank Telemarketing*, Decision Support Systems 62:22–31. UCI Machine Learning Repository, https://doi.org/10.24432/C5K306.

In [1]:
import sys
from pathlib import Path

# Make the shared module in src/ importable from this notebook
sys.path.append(str(Path.cwd().parent))

In [2]:
import pandas as pd

from src.data_utils import (load_raw, add_dates, assign_period, PERIODS,
                            LEAKAGE_COLUMNS, PROCESSED_FILE)

## 1. Read the raw file

In [3]:
df = load_raw()
print(df.shape)
df.head()

(41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


The file should contain 41,188 calls and 21 columns (20 inputs and the target `y`).

In [4]:
assert df.shape == (41188, 21), "Unexpected size: is this bank-additional-full.csv?"
df.dtypes

age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

## 2. The target and missing values

In [5]:
df["y"].value_counts(normalize=True).round(3)

y
no     0.887
yes    0.113
Name: proportion, dtype: float64

Only about 11% of the clients subscribe. The classes are unbalanced, which is why the project uses ROC-AUC and precision-recall AUC instead of accuracy.

The file has no empty cells, but some categorical columns use the label `"unknown"` for a missing value. We count them here; they are kept as their own category.

The file also contains 12 rows that are exact copies of another row. The data has no client identifier, so we cannot tell whether these are recording errors or two different calls with identical answers. They are only 0.03% of the data, so they are kept.

In [6]:
print("Empty cells:", df.isna().sum().sum())
print("Identical rows:", df.duplicated().sum())

unknown_share = (df == "unknown").mean()
unknown_share[unknown_share > 0].sort_values(ascending=False).round(3)

Empty cells: 0
Identical rows: 12


default      0.209
education    0.042
housing      0.024
loan         0.024
job          0.008
marital      0.002
dtype: float64

## 3. Rebuild the year of each call

The file only gives the month and the day of the week. The UCI documentation states that the rows are **ordered by date, from May 2008 to November 2010**. So a new year starts every time the month goes back (for example from `dec` to `mar`). The function `add_dates` in `src/data_utils.py` applies this rule.

In [7]:
df = add_dates(df)
df["year"].value_counts().sort_index()

year
2008    27690
2009    11440
2010     2058
Name: count, dtype: int64

**Check.** If the reconstruction is right, the years must run from 2008 to 2010, and the 3-month Euribor rate in each month must follow its known history: close to 5% in mid-2008, falling sharply from the end of 2008, and below 1% from mid-2009.

In [8]:
assert df["year"].min() == 2008 and df["year"].max() == 2010
assert df["year_month"].is_monotonic_increasing, "Rows are not in date order"

monthly = df.groupby("year_month").agg(
    calls=("y", "size"),
    yes_rate=("y", lambda s: (s == "yes").mean()),
    euribor_min=("euribor3m", "min"),
    euribor_max=("euribor3m", "max"),
)
monthly.round(3)

,calls,yes_rate,euribor_min,euribor_max
year_month,,,,
2008-05,7763,0.031,4.855,4.864
2008-06,4374,0.043,4.864,4.967
2008-07,6685,0.061,4.955,4.968
2008-08,5175,0.052,4.962,4.970
2008-10,67,0.627,4.760,5.045
2008-11,3616,0.053,3.853,4.733
2008-12,10,0.100,3.053,3.816
2009-03,282,0.447,1.510,1.811
2009-04,2458,0.180,1.365,1.498


The Euribor path matches its history, so the reconstructed years can be trusted.

Two other facts matter for the rest of the project:
- **Some months have no calls at all:** September 2008, January–February 2009 and January–February 2010.
- **Some months have very few calls:** October 2008 (67 calls) and December 2008 (10 calls). Their subscription rate is very noisy and should not be read on its own.

## 4. Assign the time periods

The periods are defined once, in `src/data_utils.py`:

| Period | Months | Role |
|---|---|---|
| train | May–Aug 2008 | fit the models |
| validation | Oct–Dec 2008 | tune the models; the rate has only started to fall |
| test_1 | Mar–May 2009 | just after the collapse |
| test_2 | Jun 2009–Nov 2010 | the most distant period |

In [9]:
df = assign_period(df)
assert (df["period"] != "unused").all(), "Some calls fall outside every period"

order = list(PERIODS)
summary = (df.groupby("period")
             .agg(calls=("y", "size"),
                  yes_rate=("y", lambda s: (s == "yes").mean()),
                  euribor_mean=("euribor3m", "mean"))
             .loc[order])
summary.round(3)

,calls,yes_rate,euribor_mean
period,,,
train,23997,0.046,4.923
validation,3693,0.063,4.128
test_1,8534,0.128,1.339
test_2,4964,0.445,0.874


The periods are very unequal in size: the training period holds more than half of the calls. The validation period is almost entirely November 2008 (3,616 of its 3,693 calls), when the Euribor had started to fall (3–5%) but was still far above the test periods (below 1.9%). Tuning on it is therefore a fair test of *mild* change, not of the full collapse.

## 5. Remove the leakage column and save

`duration` (the length of the call) is known only after the call has ended, so it cannot be used to decide whom to call. It is removed here, before any modelling, so that no later notebook can use it by mistake.

In [10]:
df = df.drop(columns=LEAKAGE_COLUMNS)

PROCESSED_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_FILE, index=False)
print("Saved", df.shape, "to", PROCESSED_FILE.relative_to(PROCESSED_FILE.parents[2]))

Saved (41188, 24) to data/processed/bank_with_dates.csv
